# 01 · Proxy-validity controls: does the density proxy behave as we expect, and does the established target hold up?

**What is being tested.** Whether the frozen pretrained PARNET's predicted per-nt profile recovers the
observed eCLIP signal *beyond established nulls*, on two targets:
- **(A) density proxy** - ENCODE 'signal of unique reads' bigWig (RPM read-density), recomputed live here;
- **(B) established target** - single-nt 5' crosslink **counts** built from the eCLIP BAMs (Van Nostrand 2016),
 loaded from a committed full-BAM run (BAMs are too large for the demo bundle).

**Why test it.** The density bigWig is a *surrogate* for crosslink counts. Before trusting any number on it
we must (i) expose where the surrogate is misleading, and (ii) confirm the established target genuinely
carries binding-shape signal. This is the core "do our proxies work as intended" check.

**Reasoning behind the nulls.** Our windows are peak-centered, so the observed signal is enriched in the
middle by construction. Two established nulls for autocorrelated genomic signal guard against fooling
ourselves: a **circular shift** (preserves autocorrelation) and a **center-bump** baseline (a fixed
Gaussian at the window center). Real binding shape must beat *both*, not just a trivial i.i.d. shuffle.

## Definitions: target and the established nulls

Observed signal comes in two forms. The **density proxy** is the ENCODE "signal of unique reads" bigWig,
an RPM read-density track $d\in\mathbb{R}_{\ge 0}^{L}$. The **established target** is the single-nucleotide
5' crosslink **count** vector $o\in\mathbb{Z}_{\ge 0}^{L}$ (the crosslink site is $\approx 1$ nt upstream of
the read2 5' end; Van Nostrand 2016). Both are normalized to a profile $o/\sum_i o_i$ before scoring with
Pearson $r$.

Windows are peak-**centered**, so the middle is enriched by construction. We therefore test $r$ against two
nulls that a merely centered artifact would also pass. The **circular shift**
$$r\big(\hat p,\ \operatorname{roll}(o,k)\big),\qquad k\sim\mathcal{U}\!\left[L/8,\ 7L/8\right],$$
preserves the autocorrelation of $o$; the fixed **center-bump** predictor
$$b_i=\frac{1}{Z}\exp\!\Big(-\tfrac{1}{2}\big((i-\tfrac{L-1}{2})/\sigma\big)^2\Big),\qquad \sigma=L/12,$$
is scored as $r(b,o)$. Real binding shape must beat **both** (we require $r-r_{\text{circ}}>0.05$ and
$r-r_{\text{bump}}>0.05$), not just the trivial i.i.d. shuffle.

In [ ]:
import os, sys, json, pathlib
import numpy as np
_here = pathlib.Path.cwd().resolve()
REPO = next((c for c in (_here, *_here.parents) if (c / "src" / "mmpartnet").is_dir()), _here)
sys.path.insert(0, str(REPO / "src"))
from mmpartnet import config
COMMITTED = REPO / "mmpartnet_out"          # committed result JSONs (full-BAM / prior verified runs)
RESULTS = pathlib.Path(os.environ.get("ML4RG_RESULTS", COMMITTED))   # where live recomputes are written
def load_json(name, where=COMMITTED):
    p = pathlib.Path(where) / name
    return json.loads(p.read_text()) if p.exists() else None
print("repo:", REPO, "| committed results:", COMMITTED.exists())

## Test A - density proxy vs established nulls (recomputed live on public ENCODE data)

In [ ]:
from mmpartnet.io import groups
from mmpartnet.models.parnet import load_parnet
from mmpartnet.experiments import recover_demo_profile as rdp

GROUP = os.environ.get("MMP_GROUP", "spliceosome")
CELL  = os.environ.get("MMP_CELL", "HepG2")
NWIN  = int(os.environ.get("MMP_NWIN", "20"))
manifest = json.loads((config.DATA / "eclip_manifest.json").read_text())
rbps = [g for g in (groups.resolve(GROUP) or [GROUP]) if g in manifest]
print(f"group={GROUP} cell={CELL} nwin={NWIN} RBPs={rbps}")

m = load_parnet()
dens = []
for g in rbps:
    r = rdp.recover_rbp(m, g, CELL, manifest, NWIN, target="density")
    if "error" in r:
        print(f"  {g:8} SKIP ({r['error']})"); continue
    dens.append(r)
    print(f"  {g:8} Pearson={r['pearson_mean']:+.3f}  circ={r['circular_pearson_mean']:+.3f}  "
          f"cbump={r['centerbump_pearson_mean']:+.3f}  (n={r['n_windows']})")

In [ ]:
from IPython.display import Markdown, display
def agg(rows, key): return float(np.mean([x[key] for x in rows])) if rows else float("nan")
dmp, dmc, dmb = agg(dens,"pearson_mean"), agg(dens,"circular_pearson_mean"), agg(dens,"centerbump_pearson_mean")
dens_real = (dmp-dmc>0.05) and (dmp-dmb>0.05)
display(Markdown(
f"""**Test A result (density proxy, live).** mean Pearson **{dmp:+.3f}** · circular null {dmc:+.3f} · center-bump {dmb:+.3f}
→ Pearson-circular = **{dmp-dmc:+.3f}**, Pearson-centerbump = **{dmp-dmb:+.3f}** →
{'REAL SHAPE' if dens_real else '**NOT beyond center/autocorrelation** — the density proxy is center-confounded'}.
This is the expected, honest limitation: RPM read-density blurs the crosslink site, so on peak-centerd windows a
dumb central bump explains most of it. The proxy is fine for plumbing (it runs everywhere, no BAMs) but is **not**
the target to score binding shape on."""))

## Test B - established 5' crosslink counts (committed full-BAM run)

In [ ]:
cj = load_json("recover_demo_profile_counts.json")
assert cj is not None, "recover_demo_profile_counts.json missing from committed results"
crows = [r for r in cj["rows"] if "error" not in r]
cmp_, cmc, cmb = agg(crows,"pearson_mean"), agg(crows,"circular_pearson_mean"), agg(crows,"centerbump_pearson_mean")
counts_real = (cmp_-cmc>0.05) and (cmp_-cmb>0.05)
print(f"counts target | group={cj['group']} cell={cj['cell']} target={cj['target']} | RBPs={len(crows)}")
for r in crows:
    print(f"  {r['rbp']:8} Pearson={r['pearson_mean']:+.3f}  circ={r['circular_pearson_mean']:+.3f}  cbump={r['centerbump_pearson_mean']:+.3f}")
display(Markdown(
f"""**Test B result (established counts, committed).** mean Pearson **{cmp_:+.3f}** · circular null {cmc:+.3f} · center-bump {cmb:+.3f}
→ Pearson-circular = **{cmp_-cmc:+.3f}**, Pearson-centerbump = **{cmp_-cmb:+.3f}** →
{'**REAL SHAPE (beats established nulls)**' if counts_real else 'not beyond nulls'}."""))

## Conclusion

| target | mean Pearson | vs circular | vs center-bump | verdict |
|---|---|---|---|---|
| density proxy (live) | see Test A | | | center-confounded surrogate |
| 5' crosslink counts (committed) | see Test B | | | real binding shape |

The two tests do exactly what we wanted: they **bound** the density proxy (useful as runs-everywhere plumbing,
misleading as a scoring target) and **validate** the established crosslink-count target as the one carrying real
binding-shape signal. Every density number downstream is therefore labelled a proxy; scoring claims use counts.
Claude-assisted, using the cited literature (Van Nostrand 2016; Horlacher 2023) and standard autocorrelation nulls.